In [1]:
from tensor_product_space import TensorProductSpace
import numpy as np

In [2]:
knots = [
        [0, 0, 0, 1.1, 2.9, 3, 3, 3],
        [-3,-3, -1, 0.2, 2, 2],
        #[0,0,0,1,2,3,3,3]
    ]
dim = len(knots)
p = [2, 1]
tsp = TensorProductSpace(knots, p, dim)
tsp.construct_basis()

In [3]:
def my_construct_basis(knots, dim, degrees):
    knots = np.array(knots)
    idx_start = np.array([
        list(range(len(knots[j]) - degrees[j] - 1)) for j in range(dim)
    ])

    # >>> idx_stop
    # [[4 5 6 7 8]]
    idx_stop = np.array([
        [j + degrees[i] + 2 for j in idx_start[i]] for i in range(dim)
    ])
    # All combinations of starting and stopping indices across all dimensions
    # >>> idx_start_perm
    #[[0]
    # [1]
    # [2]
    # [3]
    # [4]]
    idx_start_perm = np.stack(np.meshgrid(*idx_start), -1).reshape(-1,dim)
    idx_stop_perm = np.stack(np.meshgrid(*idx_stop), -1).reshape(-1,dim)

    n = len(idx_start_perm)
    supports_per_dim = []
    evals_per_dim = []
    basis_per_dim = []
    for j in range(dim):
        starts = knots[j][idx_start_perm[:, j]]
        ends = knots[j][idx_stop_perm[:, j]-1]
        supports_per_dim.append(np.stack((starts, ends), axis=1))

        is_at_end = idx_stop_perm[:, j] == len(knots[j])
        evals_per_dim.append(is_at_end)

        d = degrees[j]
    
        # Create the window offsets: [0, 1, ..., degree+1]
        offsets = np.arange(d + 2)
        
        # Create a grid of indices: Shape (N, degree+2)
        # We broadcast (N, 1) + (1, degree+2)
        grid_indices = idx_start_perm[:, j, None] + offsets[None, :]
        
        # Fancy index into the knot array for dimension j
        # This retrieves the slices for all N functions simultaneously
        basis_per_dim.append(knots[j][grid_indices])

    basis_supports = np.stack(supports_per_dim, axis = 1)
    basis_end_evals = np.stack(evals_per_dim, axis=1).astype(np.int32)
    basis = np.stack(basis_per_dim, axis=1)

    return np.squeeze(basis_supports), np.squeeze(basis_end_evals), np.squeeze(basis)

In [4]:
tsp.mesh.cells

array([[[ 0. ,  1.1],
        [-3. , -1. ]],

       [[ 1.1,  2.9],
        [-3. , -1. ]],

       [[ 2.9,  3. ],
        [-3. , -1. ]],

       [[ 0. ,  1.1],
        [-1. ,  0.2]],

       [[ 1.1,  2.9],
        [-1. ,  0.2]],

       [[ 2.9,  3. ],
        [-1. ,  0.2]],

       [[ 0. ,  1.1],
        [ 0.2,  2. ]],

       [[ 1.1,  2.9],
        [ 0.2,  2. ]],

       [[ 2.9,  3. ],
        [ 0.2,  2. ]]])

In [5]:
tsp.basis_supports[[3,5, 8]]

array([[[ 0. ,  1.1],
        [ 0.2,  2. ]],

       [[ 0. ,  2.9],
        [-3. ,  0.2]],

       [[ 0. ,  3. ],
        [-3. , -1. ]]])

In [7]:
support = tsp.get_cells([3,5, 8])

[[False False False False False False  True False False]
 [ True  True False  True  True False False False False]
 [ True  True  True False False False False False False]]


In [6]:
support

array([ 1,  2,  5,  6,  9, 10, 11, 14, 15, 18, 19])

In [23]:
support_cells = tsp.basis_to_cell([0,1,7])
print(tsp.mesh.cells[support_cells])

[[[ 0.   1.1]
  [-3.  -1. ]]

 [[ 1.1  2.9]
  [-3.  -1. ]]

 [[ 2.9  3. ]
  [-3.  -1. ]]

 [[ 0.   1.1]
  [-1.   0.2]]

 [[ 1.1  2.9]
  [-1.   0.2]]

 [[ 2.9  3. ]
  [-1.   0.2]]]
